In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch

# Load the model on the GPU
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map="auto",
)

# Load the processor
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

print("Model and Processor Loaded Successfully!")

In [ ]:
# im_name = "../data/inputs/IMG_0217.jpg"

# messages = [
#     {
#         "role": "user",
#         "content": [
#             {"type": "image", "image": im_name},
#             {
#                 "type": "text",
#                 "text": """Vous êtes une IA spécialisée dans l'extraction d'informations à partir de la première ligne de l'en-tête de documents historiques manuscrits en français. Votre mission est d'identifier le nom complet de la personne ainsi que le nom du mari. Identifiez uniquement le texte manuscrit.

# La première ligne suit ce format : "Nom Prénom1 (Prénom2) (Prénom3) [Marqueur] Nom_mari (Prénom_mari)"

# Voici des exemples de marqueurs : 'fme', 'fe', 'vve', 've', 'femme', 'divorcée', etc

# La sortie attendue est structurée comme suit :

# {
#   "nom_complet": "Nom Prénom",
#   "nom_complet_mari": "nom_mari (prénom_mari)",
# "marqueur": "marqueur"
# }

# Cas spécifiques :

#     La sortie ne doit pas contenir de date ou autre informations.
#     Le nom complet de l'individu se trouve obligatoirement avant le marqueur.
#     Le nom complet mari se trouve obligatoirement après le marqueur.
# """
#             },
#         ],
#     }
# ]

In [ ]:
# im_name = "../data/inputs/IMG_0152.jpg"

# messages = [
#     {
#         "role": "user",
#         "content": [
#             {"type": "image", "image": im_name},
#             {
#                 "type": "text",
#                 "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français. Votre mission est d'identifier diverses informations décrites plus loin. Identifiez uniquement le texte manuscrit.

# La première information consiste en l'identification du nom complet de la personne ainsi que le nom du mari. La première ligne suit ce format : "Nom Prénom1 (Prénom2) (Prénom3) [Marqueur] Nom_mari (Prénom_mari)"
# Voici des exemples de marqueurs : 'fme', 'fe', 'vve', 've', 'femme', 'divorcée', etc

# La sortie XML attendue est structurée comme suit :

#     <nom_complet>Nom Prénoms</nom_complet>
#     <nom_complet_mari>Nom</nom_complet_mari>
#     <marqueur>marqueur</marqueur>

# Cas spécifiques :

#     La sortie ne doit pas contenir de date ou autre informations.
#     Le nom complet de l'individu se trouve obligatoirement avant le marqueur.
#     Le nom complet mari se trouve obligatoirement après le marqueur.
#     Ecrire uniquement les informations relative aux différents champs, sans aucun autre texte.
#     N'écrire que le XML, sans aucun autre texte.

# La seconde information consiste en l'identification de la date de naissance de la personne.
# La sortie XML attendue est structurée comme suit :
    
#     <date_naissance>date</date_naissance>
#     <lieu_naissance>lieu_naissance</lieu_naissance>

# Cas spécifiques :
#     La date de naissance peut être sous la forme "jj/mm/aaaa" ou "jj mois aaaa".
#     Le lieu de naissance peut être une ville, un village, un pays, etc.
#     Le lieu de naissance peut être une ville, un village, un pays, etc se situant dans un autre pays.

# La troisième information consiste en l'identification de la nationalité de la personne.
# La sortie XML attendue est structurée comme suit :

#     <nationalite>nationalite</nationalite>

# Cas spécifiques :
#     Il est possible que la personne ait plusieurs nationalités. Dans ce cas indiquez également la date de naturalisation si elle est présente.
#     N'écrire que le XML, sans aucun autre texte.

# La quatrieme information consiste en l'identification de l'etat civil de la personne.
# La sortie XML attendue est structurée comme suit :

#     <status>status</status>
#     <adresse>adresse</adresse>
#     <ville>ville</ville>

# Cas spécifiques :
#     Le status peut être au masculin ou au féminin.
#     La ville peut être une ville, un village, un pays, etc.
#     La ville se situe obligatoirement en France.
#     Le nom de la ville est optionnel.
#     Si la ligne "Etat civil" est présente, l'adresse s'y trouve obligatoirement.
#     L'adresse doit contenir le nom de la rue et un numéro.
#     N'écrire que le JSON, sans aucun autre texte.

# La cinquième information consiste en l'identification des éléments de la table et des annotations/renseignements.
# La sortie XML attendue pour la table est structurée comme suit :

#     <table>
#         <ligne>
#             <atelier (optionnel)>element1</atelier>
#             <occupation>element2</occupation>
#             <entree>element3</entree>
#             <sortie>element4</sortie>
#             <presence><annees>X</annees><mois>Y</mois><jours (optionnel)>Z</jours></presence>
#             <observations>element5</observations>
#             <divers (optionnel)>element6</divers>
#             ...
#         </ligne>
#     </table>

# Cas spécifiques :
#     Si une cellule d'une ligne est vide ou inexistante, ecrire "".
#     Nommer les balises elements en fonction du contenu de l'entete de la colonne.
#     N'écrire que le XML, sans aucun autre texte.

# La sortie XML attendue pour les annotation est structurée comme suit :

#     <annotation>annotation</annotation>

# Cas spécifiques :
#     Il est possible que l'annotation ou le renseignement soit absent.
#     Les annotations peuvent être positionnées à différents endroits du document.
#     Les annotations peuvent tenir sur plusieurs lignes.
#     N'écrire que le XML, sans aucun autre texte.

# Pour finir il faut veiller à bien fermer les balises XML.
# """
#             },
#         ],
#     }
# ]

In [ ]:
# 0-shot

im_name = "../data/inputs/A-B/IMG_0023.jpg"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": im_name},
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous lisez les informations de l'entête du document en reconnaissant les éléments sous la forme ("first level", "second level") :
```plaintext
[("Nom complet", ""),
("Nom complet du Mari", ""),
("Marqueur", ""),
("Date de naissance", ""),
("Lieu de naissance", ""),
("Nationalité", ""),
("Statut marital", ""),
("Genre", ""),
("Adresse", ""),
("Ville", "")]
```
Contexte :
- Le document est une fiche d'entreprise manuscrite en français datant des années 1900.
- Les noms des personnes et des villes ne font pas toujours français.
- Les marqueurs peuvent être 'fme', 'fe', 'vve', 've', 'femme', 'divorcée', etc.
- Les marqueurs précèdent le nom du mari si la personne est une femme mariée.
- La date de naissance peut être sous la forme "jj/mm/aaaa" ou "jj mois aaaa".
- Le lieu de naissance peut être une ville, un village, un pays, etc.
- Un individu peut avoir plusieurs nationalités (toutes les indiquer).
- Les statuts maritaux doit être 'célibataire', 'marié', 'divorcé', 'veuf', etc ou "" si indéfini.
- Le nom de la ville est optionnel.
- Si la ligne "Etat civil" est présente, l'adresse s'y trouve obligatoirement.
- L'adresse doit contenir le nom de la rue et un numéro.	
- Le genre peut être 'homme' ou 'femme' et est déterminé par la présence d'un e à 'née'.
Tâche :
Reconstruisez, s'il vous plait, l'entête du document en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous lisez l'entête à deux niveau du tableau dans le document en reconnaissant les éléments sous la forme ("first level", "second level") :
```plaintext
[("Atelier", ""),
("Occupation", ""),
("Entrée", ""),
("Sortie", ""),
("Présence: Années", ""),
("Présence: Mois", ""),
("Présence: Jours", ""),
("Observations", ""),
("Divers", "")]
```
Contexte :
- Le document est une fiche d'entreprise textile manuscrite en français datant des années 1900.
- La colonne 'Atelier' peut être absente.
- Pour chaque ligne du tableau, créer une entrée dans le dictionnaire.
- Il est possible que certaines lignes du tableau ne contiennent qu'une observation. Dans ce cas, remplir uniquement la colonne 'Observations' d'une nouvelle entrée.
- Des lignes peuvent avoir des informations manquantes.
- Tu dois ecrire <bis> à la place de '"'.
- "Divers" contient des informations supplémentaires sur le montant de la gratifiaction par exemple (Francs et centimes).
Tâche :
Reconstruisez, s'il vous plait, le tableau en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement. Ne confond pas '"' et '11'. Tu dois egalement faire attention à ne pas confondre une ligne est une annotation marginale.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous prenez connaissance des informations du document en essayant de reperer les informations marginales sous la forme ("first level", "second level") :
```plaintext
[("Annotation", "")]
```
Contexte :
- Le document est une fiche d'entreprise textile manuscrite en français datant des années 1900.
- Les annotations sont des informations supplémentaires sur le document.
- Les annotations peuvent être des notes, des remarques, des précisions, etc.
- Les annotations sont essentiellement manuscrites.
- Les annotations peuvent être sur plusieurs lignes.
Tâche :
Reconstruisez, s'il vous plait, la liste des annotations en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement. Veillez également à ne pas mélanger les annotations.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes un agent spécialisé dans l'adaptation des informations précédemment extraites.
Votre mission est de transformer les informations extraites en XML. Pour cela vous avez a votre disposition les tags suivants :
```xml
<Document>
<Nom>
<Genre>
<Statut>
<DateDeNaissance>
<LieuDeNaissance>
<Nationalité>
<Adresse>
<Ville>
<Table>
<Ligne>
<Atelier>
<Occupation>
<Entrée>
<Sortie>
<Présence>
<Années>
<Mois>
<Jours>
<Observations>
<Divers>
<Annotation>
```
"""
            },
        ],
    }
]

In [ ]:
# Convert input into the required format
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

# Tokenize and move to GPU
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to("cuda")

In [ ]:
# Generate output
generated_ids = model.generate(**inputs, max_new_tokens=1024)

# Decode and print result
# Decode and extract only the assistant's response
output_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
assistant_response = output_text[0].split("assistant\n", 1)[-1]  # Extracts only the assistant's part

print("Generated Output:\n", assistant_response)


## Run model over batch of data

In [ ]:
import os
import re
import torch
from pathlib import Path
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
import time
from tqdm import tqdm


# Load the model on the GPU
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map="cuda",
)

# Load the processor
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

# Start timer
start_time = time.time()

input_dir = Path("../data/inputs")
output_dir = Path("../data/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Filtrer les fichiers image
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}
# Trouver toutes les images dans tous les sous-dossiers
image_files = [p for p in input_dir.rglob("*") if p.suffix.lower() in image_extensions]

system_prompts = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": None},
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous lisez les informations de l'entête du document en reconnaissant les éléments sous la forme ("first level", "second level") :
```plaintext
[("Nom", ""),
("NomDuConjoint", ""),
("DateDeNaissance", ""),
("LieuDeNaissance", ""),
("Nationalité", ""),
("Statut", ""),
("Genre", ""),
("Adresse", ""),
("Ville", "")]
```
Contexte :
- Le document est une fiche d'entreprise manuscrite en français datant des années 1900.
- Le nom de la personne est composé d'un ou plusieurs prénoms et d'un nom de famille.
- Les noms des personnes et des villes ne sont pas toujours français.
- Les marqueurs peuvent être 'fme', 'fe', 'vve', 've', 'femme', 'divorcée', etc.
- Les marqueurs précèdent le nom du mari si la personne est une femme mariée.
- Le nom du conjoint est composé d'un ou plusieurs marqueurs et du ou des noms. ex : "fme Jean Dupont", "ve Marie Curie".
- La date de naissance peut être sous la forme "jj/mm/aaaa" ou "jj mois aaaa".
- Le lieu de naissance peut être une ville, un village, un pays, etc.
- Un individu peut avoir plusieurs nationalités (toutes les indiquer).
- Les statuts maritaux doit être 'célibataire', 'marié', 'divorcé', 'veuf', etc ou "" si indéfini.
- Le nom de la ville est optionnel.
- Si la ligne "Etat civil" est présente, l'adresse s'y trouve obligatoirement.
- L'adresse doit contenir le nom de la rue et un numéro.	
- Le genre peut être 'Née' ou 'Né' et est déterminé par la présence d'un e à 'née'.
Tâche :
Reconstruisez, s'il vous plait, l'entête du document en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement. S'il y a des abréviations, ecrivez-les telles quelles.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous lisez l'entête à deux niveau du tableau dans le document en reconnaissant les éléments sous la forme ("first level", "second level") :
```plaintext
[("Atelier", ""),
("Occupation", ""),
("Entrée", ""),
("Sortie", ""),
("Présence: Années", ""),
("Présence: Mois", ""),
("Présence: Jours", ""),
("Observations", ""),
("Divers", "")]
```
Contexte :
- Le document est une fiche d'entreprise textile manuscrite en français datant des années 1900.
- La colonne 'Atelier' peut être absente.
- Pour chaque ligne du tableau, créer une entrée dans le dictionnaire.
- Il est possible que certaines lignes du tableau ne contiennent qu'une observation. Dans ce cas, remplir uniquement la colonne 'Observations' d'une nouvelle entrée.
- Des lignes peuvent avoir des informations manquantes.
- Tu dois ecrire <bis/> à la place de '"' lorsque l'on répète un élément précédemment écrit.
- "Divers" contient des informations supplémentaires sur le montant de la gratifiaction par exemple (Francs et centimes).
Tâche :
Reconstruisez, s'il vous plait, le tableau en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement. Ne confond pas '"' et '11'. Tu dois egalement faire attention à ne pas confondre une ligne est une annotation marginale.
S'il y a des abréviations, ecrivez-les telles quelles.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous prenez connaissance des informations du document en essayant de reperer les informations marginales sous la forme ("first level", "second level") :
```plaintext
[("Annotation", "")]
```
Contexte :
- Le document est une fiche d'entreprise textile manuscrite en français datant des années 1900.
- Les annotations sont des informations supplémentaires sur le document.
- Les annotations peuvent être des notes, des remarques, des précisions, etc.
- Les annotations sont essentiellement manuscrites.
- Les annotations peuvent être sur plusieurs lignes.
Tâche :
Reconstruisez, s'il vous plait, la liste des annotations en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement. Veillez également à ne pas mélanger les annotations.
S'il y a des abréviations, ecrivez-les telles quelles.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes un agent spécialisé dans l'adaptation des informations précédemment extraites.
Votre mission est de transformer les informations extraites en XML. Pour cela vous avez a votre disposition les tags suivants :
```xml
<Document>
<Nom>
<NomDuConjoint>
<Genre>
<Statut>
<DateDeNaissance>
<LieuDeNaissance>
<Nationalité>
<Adresse>
<Ville>
<Table>
<Ligne>
<Atelier>
<Occupation>
<Entrée>
<Sortie>
<Présence>
<Années>
<Mois>
<Jours>
<Observations>
<Divers>
<Annotation>
```

Si une balise est vide alors ne pas l'inclure.
"""
            },
        ],
    }
]

def extract_xml(text):
    # Recherche d'un bloc XML entre des balises markdown ```xml ... ```
    match = re.search(r"```xml\s*(.*?)\s*```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

for image_path in tqdm(image_files, desc="Processing images", unit="image"):

    torch.cuda.empty_cache()

    # Générer le chemin de sortie correspondant
    relative_path = image_path.relative_to(input_dir)
    output_subdir = output_dir / relative_path.parent
    output_subdir.mkdir(parents=True, exist_ok=True)
    output_path = output_subdir / (image_path.stem + ".xml")

    # Construire le message
    messages = system_prompts.copy()
    messages[0]["content"][0]["image"] = str(image_path)

    # Préparer le texte d'entrée pour le modèle
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=2048)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    assistant_response = output_text[0].split("assistant\n", 1)[-1]

    xml_content = extract_xml(assistant_response)
    if not xml_content:
        print(f"[⚠️] Aucun bloc XML détecté pour : {relative_path}")
        continue

    # Sauvegarde
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(xml_content)

    # Nettoyage GPU
    del generated_ids, generated_ids_trimmed, inputs, output_text, assistant_response, xml_content
    torch.cuda.empty_cache()

# Fin du traitement
end_time = time.time()
# Print time in hh:mm:ss
elapsed_time = end_time - start_time
elapsed_time_str = time.strftime("%H:%M:%S", time.gmtime(elapsed_time))
print(f"Traitement terminé en {elapsed_time_str} !")
